<a href="https://colab.research.google.com/github/Rolweezy/Calculus/blob/main/Peronalized_Recommender_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import datetime
import json
import os
from typing import Dict, List, Optional, Any
from zoneinfo import ZoneInfo

# External libraries
import math
import hashlib
import random

# If you have numpy available it's faster for vector math; fallback to pure-Python if not.
try:
    import numpy as np
    _HAS_NUMPY = True
except Exception:
    _HAS_NUMPY = False

# genai client from your original imports (kept for embedding generation if you want).
from google import genai
client = genai.Client()
from google.genai import types  # kept in-case you want to use types

# -------------------------
# Utilities: vector math
# -------------------------
def _vector_from_list(l: List[float]):
    if _HAS_NUMPY:
        return np.array(l, dtype=float)
    else:
        return l

def cosine_similarity(a: List[float], b: List[float]) -> float:
    if _HAS_NUMPY:
        a_np = np.array(a, dtype=float)
        b_np = np.array(b, dtype=float)
        denom = (np.linalg.norm(a_np) * np.linalg.norm(b_np))
        return float(np.dot(a_np, b_np) / denom) if denom != 0 else 0.0
    else:
        # pure python
        dot = sum(x*y for x,y in zip(a,b))
        norm_a = math.sqrt(sum(x*x for x in a))
        norm_b = math.sqrt(sum(x*x for x in b))
        return dot / (norm_a * norm_b) if (norm_a and norm_b) else 0.0

# -------------------------
# Mock product catalog
# -------------------------
# In production replace with real DB fetch including precomputed vector embeddings for each product.
SAMPLE_PRODUCT_CATALOG = [
    {
        "id": "p1001",
        "title": "Minimalist White Sneakers",
        "price": 79.99,
        "category": "fashion",
        "target_demographic": {"gender": "unisex", "age_min": 18, "age_max": 45},
        "image_url": "https://example.com/img/p1001.jpg",
        "rating": 4.6,
        # embedding: precomputed vector (length 8 in mock); production: 768+ dims
        "embedding": [0.15, 0.02, 0.5, 0.33, 0.12, 0.02, 0.01, 0.05],
        "tags": ["sneakers", "casual", "comfortable"],
    },
    {
        "id": "p1002",
        "title": "Noise-Cancelling Headphones",
        "price": 199.99,
        "category": "electronics",
        "target_demographic": {"gender": "unisex", "age_min": 16, "age_max": 65},
        "image_url": "https://example.com/img/p1002.jpg",
        "rating": 4.8,
        "embedding": [0.02, 0.7, 0.1, 0.05, 0.44, 0.12, 0.01, 0.03],
        "tags": ["audio", "work", "travel"],
    },
    {
        "id": "p1003",
        "title": "Organic Skincare Set",
        "price": 49.50,
        "category": "beauty",
        "target_demographic": {"gender": "female", "age_min": 20, "age_max": 60},
        "image_url": "https://example.com/img/p1003.jpg",
        "rating": 4.5,
        "embedding": [0.3, 0.05, 0.1, 0.48, 0.07, 0.02, 0.01, 0.2],
        "tags": ["skincare", "organic", "selfcare"],
    },
    # ... extend catalog
]

# -------------------------
# Embedding utility
# -------------------------
def get_user_embedding_from_signals(user_profile: Dict[str, Any], use_genai=False) -> List[float]:
    """
    Build a user embedding vector from signals (browsing history, purchases, preferences, mood, demographics).
    - If use_genai True, attempt to call genai embeddings API on a textual aggregation of signals.
    - Otherwise create a deterministic pseudo-embedding using hashing (good for deterministic reranking tests).
    """
    # Aggregate a textual representation
    pieces = []
    keys = ["browsing_history", "past_purchases", "preferences", "mood", "demographics", "budget"]
    for k in keys:
        v = user_profile.get(k)
        if v:
            pieces.append(json.dumps(v, sort_keys=True))
    aggregate = " ||| ".join(pieces)[:3000]  # limit length

    if use_genai:
        try:
            # Example call — adapt to your actual genai embeddings API method
            emb_resp = client.embeddings.create(model="embed-text-001", input=aggregate)
            # emb_resp.data[0].embedding typical shape; adapt if API differs
            return list(emb_resp.data[0].embedding)
        except Exception as e:
            # fallback
            print("genai embedding failed, falling back to pseudo-embedding:", e)

    # Deterministic pseudo-embedding fallback (stable across runs for same profile)
    # Create a vector of floats from a sha256 hash
    digest = hashlib.sha256(aggregate.encode("utf-8")).hexdigest()
    # convert hex chunks to numbers
    vec = []
    chunk_size = 8
    for i in range(0, min(len(digest), 128), chunk_size):
        chunk = digest[i:i+chunk_size]
        num = int(chunk, 16)
        # normalize to 0..1
        vec.append((num % 10000) / 10000.0)
    # ensure minimal length (match product embedding dim in catalog for demo)
    target_len = len(SAMPLE_PRODUCT_CATALOG[0]["embedding"])
    if len(vec) < target_len:
        while len(vec) < target_len:
            vec.append(random.random() * 0.01)
    return vec[:target_len]

# -------------------------
# Deterministic reranker
# -------------------------
def deterministic_rerank(
    user_embedding: List[float],
    product_candidates: List[Dict[str, Any]],
    user_profile: Dict[str, Any],
    weights: Optional[Dict[str, float]] = None
) -> List[Dict[str, Any]]:
    """
    Rerank candidate products deterministically using:
      - cosine similarity between user_embedding and product.embedding
      - hard rules: budget penalty/boost, demographic match boost, purchase-diversity penalty, mood boosts
      - combine into a final score and sort descending
    Returns products annotated with 'score' and 'reason_breakdown'.
    """
    if weights is None:
        weights = {
            "emb_sim": 1.0,
            "budget": 1.5,
            "demographic": 0.3,
            "purchase_diversity": 0.5,
            "mood": 0.6,
        }

    budget = user_profile.get("budget")  # expected numeric or None
    purchases = user_profile.get("past_purchases", [])
    mood = user_profile.get("mood", "").lower()
    demographics = user_profile.get("demographics", {})

    ranked = []
    for p in product_candidates:
        base_sim = cosine_similarity(user_embedding, p.get("embedding", [0]*len(user_embedding)))
        score = weights["emb_sim"] * base_sim
        reasons = {"emb_sim": base_sim}

        # Budget: penalize if product price > budget; small boost if comfortably within budget
        price = p.get("price", 0.0)
        budget_score = 0.0
        if budget is not None:
            try:
                b = float(budget)
                if price > b:
                    # penalty increases with how much over budget
                    over_ratio = (price - b) / (b + 1e-6)
                    budget_score = -min(1.0, over_ratio)  # cap penalty at -1
                else:
                    # boost: how close to max budget (user likely to spend near top of budget)
                    within_ratio = 1 - (b - price) / (b + 1e-6)
                    budget_score = max(0.0, within_ratio) * 0.5  # smaller boost
            except Exception:
                budget_score = 0.0
        score += weights["budget"] * budget_score
        reasons["budget"] = budget_score

        # Demographic match: boost if target demographic fits
        demo_score = 0.0
        try:
            targ = p.get("target_demographic", {})
            user_age = demographics.get("age")
            user_gender = demographics.get("gender", "").lower()
            if user_age is not None:
                amin = targ.get("age_min")
                amax = targ.get("age_max")
                if amin is None or amax is None or (amin <= user_age <= amax):
                    demo_score += 0.2
            targ_gender = targ.get("gender", "unisex").lower()
            if targ_gender == "unisex" or targ_gender == user_gender or user_gender == "":
                demo_score += 0.2
        except Exception:
            demo_score = 0.0
        score += weights["demographic"] * demo_score
        reasons["demographic"] = demo_score

        # Purchase diversity: if user already purchased this exact product recently, lower score (unless it's consumable)
        diversity_penalty = 0.0
        if any(p["id"] == prev.get("id") for prev in purchases if isinstance(prev, dict)):
            diversity_penalty = -0.7
        else:
            # if user purchased many items in same category, penalize near-duplicates to encourage variety
            purchased_categories = {prev.get("category") for prev in purchases if isinstance(prev, dict)}
            if p.get("category") in purchased_categories:
                diversity_penalty = -0.15
        score += weights["purchase_diversity"] * diversity_penalty
        reasons["purchase_diversity"] = diversity_penalty

        # Mood-based boosts: map mood words to categories/tags and boost
        mood_score = 0.0
        mood_map = {
            "happy": ["celebration", "fashion", "accessories"],
            "sad": ["selfcare", "comfort", "books"],
            "stressed": ["audio", "wellness", "relaxation"],
            "excited": ["gadgets", "games", "adventure"],
            "bored": ["hobby", "books", "games"],
            "romantic": ["beauty", "fragrance", "jewelry"],
        }
        mood_targets = mood_map.get(mood, [])
        # if product has tags matching mood targets, small boost
        if mood_targets:
            if any(tag in mood_targets for tag in p.get("tags", [])):
                mood_score = 0.25
        score += weights["mood"] * mood_score
        reasons["mood"] = mood_score

        # Add small popularity/rating factor (deterministic)
        rating = p.get("rating", 4.0)
        rating_score = (rating - 3.5) / 1.5  # map rating ~3.5..5 -> ~0..1
        score += 0.1 * rating_score
        reasons["rating_boost"] = rating_score

        # Final aggregation
        ranked.append({
            "product": p,
            "score": score,
            "reason_breakdown": reasons
        })

    # deterministic sort (stable) by score desc then product id to ensure repeatability
    ranked_sorted = sorted(ranked, key=lambda x: (-x["score"], x["product"]["id"]))
    return ranked_sorted

# -------------------------
# Chatbot UI payload generator
# -------------------------
def generate_chatbot_ui_payload(recommendation: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convert a single recommendation into a chatbot UI payload (JSON) with:
    - card: title, subtitle, image, price, rating
    - actions: Buy, Add to cart, More like this, Save for later
    - quick replies
    """
    p = recommendation["product"]
    score = recommendation["score"]
    reasons = recommendation["reason_breakdown"]

    title = p.get("title")
    subtitle = f"${p.get('price'):.2f} • {p.get('category').title()} • Rating {p.get('rating')}"
    description = f"Recommended score {score:.3f}. Top reasons: " + ", ".join(
        [f"{k}:{(v if isinstance(v, float) else str(v)):.2f}" if isinstance(v, float) else f"{k}:{v}"
         for k,v in reasons.items()]
    )

    payload = {
        "type": "product_card",
        "id": p.get("id"),
        "title": title,
        "subtitle": subtitle,
        "image_url": p.get("image_url"),
        "price": p.get("price"),
        "rating": p.get("rating"),
        "description": description,
        "actions": [
            {"type": "postback", "title": "Buy now", "payload": json.dumps({"action": "buy", "product_id": p["id"]})},
            {"type": "postback", "title": "Add to cart", "payload": json.dumps({"action": "add_to_cart", "product_id": p["id"]})},
            {"type": "postback", "title": "More like this", "payload": json.dumps({"action": "more_like", "product_id": p["id"]})},
            {"type": "postback", "title": "Save for later", "payload": json.dumps({"action": "save", "product_id": p["id"]})},
        ],
        "quick_replies": [
            {"title": "Show similar", "payload": json.dumps({"action": "more_like", "product_id": p["id"]})},
            {"title": "Open product", "payload": json.dumps({"action": "open", "product_id": p["id"]})},
            {"title": "Not interested", "payload": json.dumps({"action": "dismiss", "product_id": p["id"]})}
        ]
    }
    return payload

# -------------------------
# High-level function: produce recommendations + UI payloads
# -------------------------
def generate_hyper_personalized_recommendations(
    user_profile: Dict[str, Any],
    catalog: Optional[List[Dict[str, Any]]] = None,
    top_k: int = 3,
    use_genai_embedding: bool = False
) -> Dict[str, Any]:
    """
    Main entrypoint:
    - build user embedding
    - pick candidate products from catalog (simple filtering + shortlisting)
    - rerank deterministically
    - return top_k recommendations and example chatbot UI payloads
    """
    if catalog is None:
        catalog = SAMPLE_PRODUCT_CATALOG

    # 1) Build user embedding
    user_emb = get_user_embedding_from_signals(user_profile, use_genai=use_genai_embedding)

    # 2) Candidate selection (simple): filter by categories/preferences + budget window
    preferences = user_profile.get("preferences", [])
    budget = user_profile.get("budget")
    candidates = []
    for p in catalog:
        # simple preference matching: include if any preferred category or tag appears, or include all if no prefs
        include = True
        if preferences:
            include = bool(set(preferences).intersection(set([p.get("category")] + p.get("tags", []))))
        # budget filter: optionally exclude extremely expensive items (e.g., > 3x budget)
        if budget:
            try:
                b = float(budget)
                if p.get("price", 0) > 3 * b:
                    include = False
            except Exception:
                pass
        if include:
            candidates.append(p)

    # fallback: if no candidates due to strict filters, relax and use whole catalog
    if not candidates:
        candidates = catalog.copy()

    # 3) Deterministic reranker
    ranked = deterministic_rerank(user_emb, candidates, user_profile)

    # 4) Format outputs & build UI payloads
    top = ranked[:top_k]
    recommendations = []
    for r in top:
        p = r["product"]
        rec = {
            "product_id": p["id"],
            "title": p["title"],
            "price": p["price"],
            "category": p["category"],
            "score": r["score"],
            "reason_breakdown": r["reason_breakdown"]
        }
        ui_payload = generate_chatbot_ui_payload(r)
        rec["chatbot_ui_payload"] = ui_payload
        recommendations.append(rec)

    return {
        "status": "success",
        "generated_at": datetime.datetime.now(tz=ZoneInfo("UTC")).isoformat(),
        "recommendations": recommendations
    }

# -------------------------
# Example usage (demo)
# -------------------------
if __name__ == "__main__":
    demo_user = {
        "browsing_history": ["p1002", "p1005", "audio books page"],
        "past_purchases": [{"id": "p0987", "category": "electronics"}, {"id":"p1005","category":"books"}],
        "budget": 150.00,
        "demographics": {"age": 29, "gender": "male"},
        "preferences": ["electronics", "audio", "gadgets"],
        "mood": "stressed"
    }

    out = generate_hyper_personalized_recommendations(demo_user, top_k=3, use_genai_embedding=False)
    print(json.dumps(out, indent=2))


ValueError: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.